## **Fine-Tuning mBERT for Sentiment Analysis**
Created by Genevieve Powell

---

Sources:

*   FineTuning BERT for Arabic Sentiment Analysis: https://colab.research.google.com/github/khaledsoudy-1/FineTuning-BERT-Arabic-Sentiment/blob/main/Fine_Tuning_BERT.ipynb#scrollTo=1wVm7ZxozkRG
*   Recent FineTuning application for BERT: https://dev.to/nareshnishad/fine-tuning-bert-unlocking-the-power-of-pre-trained-language-models-4i1k


In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 98.1 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1


In [ ]:
#importing mBERT model for sentiment analysis
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "PRAli22/AraBert-Arabic-Sentiment-Analysis"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, ignore_mismatched_sizes=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#import TUNIZI Data from Google Drive
file_path = "/content/drive/MyDrive/CompNeuro Project/TUNIZI_V2.txt"

In [ ]:
#Creating df from data file
import pandas as pd

df = pd.read_csv(file_path, sep=';', header=None, names=["text", "label"])
df

In [ ]:
#Generating five random lines, random_state=41 means the same five will be chosen each time
random_5_rows = df.sample(n=5, random_state=41)
random_5_rows

In [ ]:
#Copy five random rows to avoid modifying the original dataset
five_rows_copy = random_5_rows.copy()

#BERT sentiment models automatically have five sentiment categories, map to three (Positive, Neutral, Negative)
custom_mapping = {
    0: -1,
    1: -1,
    2: 0,
    3: 1,
    4: 1
}

In [ ]:
#Testing the mBERT model on the dataset
import numpy as np
import torch

predicted_sentiments_list = []

for row in five_rows_copy['text']:
  inputs = tokenizer(row, return_tensors="pt")

  #Move inputs to the same device as the model
  inputs = {k: v.to(model.device) for k, v in inputs.items()}

  #Run model inference
  with torch.no_grad():
    outputs = model(**inputs)

  #Get logits and move to CPU
  logits = outputs.logits.detach().cpu().numpy()

  #Find the predicted class index
  predicted_class = np.argmax(logits, axis=-1)[0]

  #Map to custom sentiment label (handling out of bounds index if needed)
  predicted_sentiment = custom_mapping.get(predicted_class, predicted_class)

  #Append to sentiments list
  predicted_sentiments_list.append(predicted_sentiment)

print(predicted_sentiments_list)

In [ ]:
#Comparing the random sample labels to the sentiment predictions before fine-tuning
five_rows_copy['Bert_Sentiments'] = predicted_sentiments_list
five_rows_copy

In [ ]:
df.shape

In [ ]:
#Randomly sample 30000 out of 10000 TUNIZI comments, for shorter runtime
df = df.sample(n=30000, random_state=42)

print(f"Total Number of Rows and Columns After Sampling: {df.shape}")

In [ ]:
#Print unique labels in the dataset
import numpy as np

for label in np.unique(df['label']):
  print(label)

In [ ]:
#Changes -1:Negative, 0:Neutral, 1:Positive labels to sequential 0, 1, 2 sentiment labels
from sklearn.preprocessing import LabelEncoder

LE = LabelEncoder()

df['label'] = LE.fit_transform(df['label'])
df.head()

In [ ]:
#Check after converting labels into numerics
for label in np.unique(df['label']):
  print(label)

In [ ]:
#Splits training dataset into training and validation
from sklearn.model_selection import train_test_split

train, val = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
#Check the new shapes
print(train.shape)
print(val.shape)

In [ ]:
#Checking sets
train.head()

In [ ]:
#Reseting the index values
train.reset_index(drop=True, inplace=True)
val.reset_index(drop=True, inplace=True)

train.head()

In [ ]:
train.head()

In [ ]:
val.head()

In [ ]:
#Converts text and label columns of the traing data into Numpy arrays
x_train = train['text'].to_numpy()
y_train = train['label'].to_numpy()

#Apply to the validation data
x_test = val['text'].to_numpy()
y_test = val['label'].to_numpy()

In [ ]:
print(f"Text column of the Train Data:\n {x_train}")
print(f"\nLabel column of the Train Data:\n {y_train}")
print(type(x_train))
print(type(y_train))

In [ ]:
print(f"Text column of the Validation (Test) Data:\n {x_test}")
print(f"\nLabel column of the Validation (Test) Data:\n {y_test}")

In [ ]:
from torch.utils.data import Dataset
import torch

#Defines a class to hold the dataset and tokenizer
class SentimentDataset(Dataset):

    #Initialization step
    def __init__(self, texts, labels, tokenizer):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer

    #Returns length of sentiment text lengths
    def __len__(self):
        return len(self.texts)

    #Returns text, label, and encodings
    def __getitem__(self, idx):

        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(text, padding='max_length', max_length=128, truncation=True,return_tensors='pt')

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
#Creates training and evaluation SentimentDataset objects
train_dataset = SentimentDataset(x_train, y_train, tokenizer)

eval_dataset = SentimentDataset(x_test, y_test, tokenizer)

In [ ]:
#Sets up training configuration (batch size, learning rate, etc.) and trainer model object
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,

    logging_steps=100,

    eval_strategy='epoch',
    save_strategy='epoch',

    report_to='none'
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=eval_dataset)

In [ ]:
#Training
trainer.train()

In [ ]:
#Evaluation (Loss)
trainer.evaluate()

In [ ]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

#Runs the trained model on validation dataset
results = trainer.predict(eval_dataset)

#Converts model outputs into predicted class labels
y_pred = np.argmax(results.predictions, axis=1)

#Gets the actual labels
y_true = results.label_ids

In [ ]:
#Prints Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

print("Confusion Matrix:")
print(cm)

In [ ]:
#Prints Classification Report (Precision, recall, f-1 score, tec.)
print(classification_report(
    y_true,
    y_pred,
    target_names=['Negative', 'Neutral', 'Positive']
))